# Reading glTF scenes

`k3d.glb()` reads a binary glTF (`.glb`), `k3d.gltf()` the JSON form (`.gltf`, with its buffers
and images next to it or inlined). Both return a `k3d.objects.Group` of ordinary K3D objects -
one `k3d.mesh` per glTF primitive, `k3d.points` and `k3d.lines` for point and line primitives -
so every part of a model has its own controls in the panel, is saved in snapshots, and can be
changed from Python like anything else.

The models below are from the Khronos glTF Sample Assets
(https://github.com/KhronosGroup/glTF-Sample-Assets), each under the licence named next to it.

In [ ]:
import numpy as np

import k3d
from k3d.helpers import download

SAMPLES = 'https://raw.githubusercontent.com/KhronosGroup/glTF-Sample-Assets/main/Models/'

## A model of several parts

The milk truck (Cesium, CC-BY 4.0) has a body, glass, trim and two wheel sets. Each is a mesh
named after its glTF node, and all of them share one `group` - the file name - so the panel
shows one folder with a sub-folder per part.

glTF is Y-up and K3D Z-up: by default the model is turned upright (`up='y'`). Pass `up='z'`
for a file that is already Z-up, such as K3D's own glTF export.

In [ ]:
truck = k3d.glb(download(SAMPLES + 'CesiumMilkTruck/glTF-Binary/CesiumMilkTruck.glb'))

for part in truck:
    print(f'{part.name:28} {part.type:6} {part.vertices.shape[0]:5} vertices')

In [ ]:
plot = k3d.plot(renderer='advanced')
plot += truck
plot.display()

## The parts are ordinary objects

Change one by name, or all of them in a loop. The group itself has no properties - setting one
on it raises, rather than silently doing nothing - with one exception: `model_matrix` moves the
whole model. The node hierarchy of the file is kept as `k3d.transform.Transform`s under
`truck.transform`, so every part stays where the file put it.

In [ ]:
truck['Cesium_Milk_Truck/glass'].opacity = 0.4

In [ ]:
truck.model_matrix = k3d.transform(rotation=[np.pi / 4, 0, 0, 1], translation=[0, 0, 1]).model_matrix

In [ ]:
try:
    truck.opacity = 0.5
except AttributeError as e:
    print(e)

## Materials

The material of each primitive becomes mesh parameters: `color` and `opacity` from the base
colour, `texture`, `metalness_roughness_map`, `normal_map`, `occlusion_map`, `emissive` with
`emissive_map`, `alpha_mode` and `alpha_cutoff`, `side`, and `texture_wrap`. Colour factors are
stored linear in glTF and converted to the displayed values K3D uses.

The boombox (Microsoft, CC0) uses all of them - its PLAY button is emissive.

In [ ]:
boombox = k3d.glb(download(SAMPLES + 'BoomBox/glTF-Binary/BoomBox.glb'), scaling=[100, 100, 100])
part = boombox[0]

for name in ('texture', 'normal_map', 'metalness_roughness_map', 'occlusion_map', 'emissive_map'):
    print(f'{name:24} {len(getattr(part, name)) / 1e6:5.1f} MB')

print('emissive', hex(part.emissive), 'alpha_mode', part.alpha_mode)

In [ ]:
plot = k3d.plot(renderer='advanced', environment='studio', tone_mapping='aces', grid_visible=False,
                background_color=0x1E2126)
plot += boombox
plot.display()

The same scene path traced: the emissive button lights what is in front of it, and the
chrome reflects the environment. Every map the tracer reads (colour, normal, metalness-roughness,
emissive, alpha) reaches it; occlusion it computes itself.

In [ ]:
plot.renderer = 'cinematic'
plot.cinematic_samples = 64
plot.cinematic_denoise = 0.8

## Compressed geometry, glass and gems

The ring ("Sasha" by saber7711 on Blendswap, CC-BY; converted to glTF for the K3D docs, see
`docs/source/gallery/showcase/assets/sasha.md`) is Draco-compressed - reading it needs `pip install DracoPy` - and its stones are transmissive,
with an index of refraction of 2.09. `transmission`, `ior`, `thickness` and the attenuation are
mesh parameters, and the cinematic renderer refracts through them.

Every part keeps the name of its glTF material in `custom_data`, which is how the band is
found and restyled here.

In [ ]:
ring = k3d.glb('../docs/source/gallery/showcase/assets/sasha.glb', rotation=[np.pi / 4, 0, -1, 0])

for part in ring:
    if part.custom_data.get('gltf_material') == 'Material.002':
        part.color = 0xD4A27F   # rose gold
        part.roughness = 0.05

print(sorted({(part.custom_data['gltf_material'], part.transmission, round(part.ior, 2), part.thickness)
              for part in ring}))

Left to fit itself, the camera frames the plot's grid, rounded out to whole ticks, so a
small object ends up small. Here it is placed instead: on the stones, from outside the band. A
floor catches the shadow, and a glowing panel overhead - out of frame - is the light the stones
sparkle with.

In [ ]:
boxes = np.array([part.get_bounding_box() for part in ring])
low, high = boxes[:, 0::2].min(axis=0), boxes[:, 1::2].max(axis=0)
centre, size = (low + high) / 2, float((high - low).max())


def quad(z, half, **kwargs):
    corners = [[centre[0] + sx * half, centre[1] + sy * half, z] for sx, sy in ((-1, -1), (1, -1), (1, 1), (-1, 1))]
    return k3d.mesh(np.array(corners, np.float32), np.array([[0, 2, 1], [0, 3, 2]], np.uint32), side='double', **kwargs)


plot = k3d.plot(renderer='cinematic', environment='brown_photostudio_02', tone_mapping='aces', lighting=2.0,
                grid_visible=False, camera_auto_fit=False, camera_fov=30, background_color=0xE6E6E6,
                cinematic_samples=64, cinematic_denoise=0.8, cinematic_bounces=32)
plot += ring
plot += quad(float(low[2]), 3 * size, color=0xF2F2F2, roughness=0.5, name='floor')
plot += quad(float(high[2]) + 1.5 * size, size, color=0x000000, emissive=0xFFFFFF, emissive_intensity=4.0,
             name='light box')

gems = np.array([part.get_bounding_box() for part in ring if part.transmission > 0])
target = (gems[:, 0::2].min(axis=0) + gems[:, 1::2].max(axis=0)) / 2
out = (target - centre) * [1, 1, 0]
out /= np.linalg.norm(out)
eye = target + (0.9 * out - 0.75 * np.cross([0, 0, 1], out) + [0, 0, 1.0]) * size
plot.camera = [*eye, *target, 0, 0, 1]

plot.display()

## A .gltf with files beside it

A `.gltf` is read from its path: the reader finds its `.bin` and images relative to it. Its
bytes work too, as long as everything is inlined as `data:` URIs.

In [ ]:
box = k3d.gltf('../docs/source/reference/assets/factory/BoxTextured/BoxTextured.gltf')

plot = k3d.plot()
plot += box
plot.display()

## What is not shown

Animations, skins (the vertices are shown as stored), cameras, lights, KTX2 textures, and
material extensions beyond emissive strength, unlit, transmission, ior and volume. A file using
any of them is still read, and one warning names what was left out; a file that *requires*
meshopt compression raises, with a hint how to decompress it.

In [ ]:
chair = k3d.glb(download(SAMPLES + 'SheenChair/glTF-Binary/SheenChair.glb'))

## The material parameters on any mesh

The parameters glTF needed are plain `k3d.mesh` parameters, so a mesh built from arrays takes
them too. Since 3.2.0 `color` multiplies `colors`, the colormap and the texture; left out, it is
white whenever one of those is given.

In [ ]:
import io

from PIL import Image


def png(pixels):
    buffer = io.BytesIO()
    Image.fromarray(np.asarray(pixels, np.uint8)).save(buffer, format='PNG')
    return buffer.getvalue()


vertices = np.array([[0, 0, 0], [1, 0, 0], [1, 0, 1], [0, 0, 1]], np.float32)
indices = np.array([[0, 1, 2], [0, 2, 3]], np.uint32)
uvs = np.array([[0, 1], [1, 1], [1, 0], [0, 0]], np.float32)

y, x = (np.mgrid[0:64, 0:64] + 0.5) / 64 - 0.5
disc = np.dstack([np.full((64, 64), 255), np.full((64, 64), 140), np.zeros((64, 64)),
                  np.clip(1 - np.hypot(x, y) / 0.5, 0, 1) * 255])

plot = k3d.plot()
# alpha of the texture: ignored, blended, cut at a threshold
plot += k3d.mesh(vertices, indices, uvs=uvs, texture=png(disc), alpha_mode='opaque')
plot += k3d.mesh(vertices + [1.1, 0, 0], indices, uvs=uvs, texture=png(disc), alpha_mode='blend')
plot += k3d.mesh(vertices + [2.2, 0, 0], indices, uvs=uvs, texture=png(disc), alpha_mode='mask')
# RGBA per vertex: the fourth column becomes opacities and makes the mesh blended
plot += k3d.mesh(vertices + [0, 0, -1.1], indices,
                 colors=np.array([[1, 0, 0, 1], [0, 1, 0, 1], [0, 0, 1, 0], [1, 1, 0, 0]], np.float32))
# a glowing tile
plot += k3d.mesh(vertices + [1.1, 0, -1.1], indices, color=0x101010, emissive=0x40C0FF,
                 emissive_intensity=2.0)
plot.display()